# APP Training Runner
This notebook runs the Chapter 4 training pipeline on Google Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))


In [ ]:
%cd /content
!git clone <GITHUB-REPOSITORY-URL> thesis-app-training
%cd /content/thesis-app-training
!bash scripts/colab_setup.sh


In [ ]:
!mkdir -p /content/data
!rsync -a --info=progress2 "/content/drive/MyDrive/thesis_app/image_output/" /content/data/image_output/


In [ ]:
import os
os.environ['MANIFEST'] = '/content/data/image_output/image_manifest.csv'
os.environ['IMAGE_ROOT'] = '/content/data/image_output'
os.environ['RUN_ROOT'] = '/content/drive/MyDrive/thesis_app/app_runs'


In [ ]:
!bash scripts/run_smoke_test.sh "$MANIFEST" "$IMAGE_ROOT" "$RUN_ROOT"


Run models one at a time for better control.


In [ ]:
!python src/train.py --config configs/resnet50.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --run-dir "$RUN_ROOT/resnet50"
!python src/evaluate.py --config configs/resnet50.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --checkpoint "$RUN_ROOT/resnet50/checkpoints/best.keras" --out-dir "$RUN_ROOT/resnet50/eval_validation" --split validation
!python src/evaluate.py --config configs/resnet50.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --checkpoint "$RUN_ROOT/resnet50/checkpoints/best.keras" --out-dir "$RUN_ROOT/resnet50/eval_test" --split test


In [ ]:
!python src/train.py --config configs/efficientnetb0.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --run-dir "$RUN_ROOT/efficientnetb0"
!python src/evaluate.py --config configs/efficientnetb0.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --checkpoint "$RUN_ROOT/efficientnetb0/checkpoints/best.keras" --out-dir "$RUN_ROOT/efficientnetb0/eval_validation" --split validation
!python src/evaluate.py --config configs/efficientnetb0.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --checkpoint "$RUN_ROOT/efficientnetb0/checkpoints/best.keras" --out-dir "$RUN_ROOT/efficientnetb0/eval_test" --split test


In [ ]:
!python src/train.py --config configs/convnext_tiny.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --run-dir "$RUN_ROOT/convnext_tiny"
!python src/evaluate.py --config configs/convnext_tiny.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --checkpoint "$RUN_ROOT/convnext_tiny/checkpoints/best.keras" --out-dir "$RUN_ROOT/convnext_tiny/eval_validation" --split validation
!python src/evaluate.py --config configs/convnext_tiny.yaml --manifest "$MANIFEST" --image-root "$IMAGE_ROOT" --checkpoint "$RUN_ROOT/convnext_tiny/checkpoints/best.keras" --out-dir "$RUN_ROOT/convnext_tiny/eval_test" --split test


In [ ]:
!python src/compare_runs.py --eval-root "$RUN_ROOT" --split validation --out "$RUN_ROOT/comparison_validation.csv"
!python src/compare_runs.py --eval-root "$RUN_ROOT" --split test --out "$RUN_ROOT/comparison_test.csv"
